In [3]:
import pandas as pd
from sqlalchemy import create_engine
import urllib.parse

# 1. SQL Server Bağlantı Ayarları
SERVER_NAME = r'DERYAZADE1515\SQLEXPRESS' # SSMS'te sunucuya bağlanırken kullandığın adı buraya yaz
DATABASE_NAME = 'Satis_Analiz_1' # Veritabanı adın

# 2. Windows Authentication (Şifresiz Güvenli Giriş) Bağlantı Dizesi
params = urllib.parse.quote_plus(
    f'Driver={{SQL Server}};'
    f'Server={SERVER_NAME};'
    f'Database={DATABASE_NAME};'
    f'Trusted_Connection=yes;'
)

# 3. SQLAlchemy Motorunu Oluşturma
engine = create_engine(f'mssql+pyodbc:///?odbc_connect={params}')

# 4. Veriyi SQL'den Pandas DataFrame'e Çekme
query = "SELECT * FROM superstore_sales;"
df = pd.read_sql(query, engine)

# 5. Verinin Başarıyla Çekildiğini Test Etme (İlk 5 Satırı Göster)
df.head()


,Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,...,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales,Quantity,Discount,Profit
0,1,CA-2013-152156,2013-11-09,2013-11-12,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,26196.0,2,0.0,419136.0
1,2,CA-2013-152156,2013-11-09,2013-11-12,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",73194.0,3,0.0,219582.0
2,3,CA-2013-138688,2013-06-13,2013-06-17,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,1462.0,2,0.0,68714.0
3,4,US-2012-108966,2012-10-11,2012-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,9575775.0,5,45.0,-383031.0
4,5,US-2012-108966,2012-10-11,2012-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22368.0,2,2.0,25164.0


In [4]:
# 1. Satır/Sütun sayısı ve Veri Tipleri
print("--- VERİ TİPLERİ VE SATIR BİLGİSİ ---")
df.info()

print("\n", "="*50, "\n")

# 2. Hangi sütunda kaç tane eksik (NULL/NaN) veri var?
print("--- EKSİK VERİ KONTROLÜ ---")
print(df.isnull().sum())

--- VERİ TİPLERİ VE SATIR BİLGİSİ ---
<class 'pandas.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row_ID         9994 non-null   int64  
 1   Order_ID       9994 non-null   str    
 2   Order_Date     9994 non-null   str    
 3   Ship_Date      9994 non-null   str    
 4   Ship_Mode      9994 non-null   str    
 5   Customer_ID    9994 non-null   str    
 6   Customer_Name  9994 non-null   str    
 7   Segment        9994 non-null   str    
 8   Country        9994 non-null   str    
 9   City           9994 non-null   str    
 10  State          9994 non-null   str    
 11  Postal_Code    9994 non-null   str    
 12  Region         9994 non-null   str    
 13  Product_ID     9994 non-null   str    
 14  Category       9994 non-null   str    
 15  Sub_Category   9994 non-null   str    
 16  Product_Name   9994 non-null   str    
 17  Sales          9994 non-n

In [5]:
# 1. Metinleri gerçek 'datetime' formatına çeviriyoruz
df['Order_Date'] = pd.to_datetime(df['Order_Date'])
df['Ship_Date'] = pd.to_datetime(df['Ship_Date'])

# 2. EKSİK VERİ: Sadece 1 satır olduğu için, o satırı veri setinden kalıcı olarak siliyoruz.
# (Eğer binlerce boşluk olsaydı silmek yerine ortalama değer ile dolduracaktık)
df.dropna(subset=['Profit'], inplace=True)

# 3. KONTROL: 
print("--- YENİ VERİ TİPLERİ ---")
print(df[['Order_Date', 'Ship_Date']].dtypes)

print("\n--- YENİ EKSİK VERİ DURUMU ---")
print("Kâr Sütunundaki Boşluk Sayısı:", df['Profit'].isnull().sum())
print("Güncel Toplam Satır Sayısı:", len(df))

--- YENİ VERİ TİPLERİ ---
Order_Date    datetime64[us]
Ship_Date     datetime64[us]
dtype: object

--- YENİ EKSİK VERİ DURUMU ---
Kâr Sütunundaki Boşluk Sayısı: 0
Güncel Toplam Satır Sayısı: 9993


In [6]:
# 1. Tekrar Eden (Kopya/Duplicate) Satır Kontrolü
print("--- KOPYA SATIR KONTROLÜ ---")
kopya_sayisi = df.duplicated().sum()
print(f"Tekrar eden satır sayısı: {kopya_sayisi}")

# Eğer kopya varsa otomatik silmesi için önlem
if kopya_sayisi > 0:
    df.drop_duplicates(inplace=True)
    print("Kopya satırlar başarıyla silindi ve veri temizlendi.")

# 2. Mantıksal Hata Kontrolü (Negatif Satış veya Miktar Olamaz)
print("\n--- MANTIKSAL HATA KONTROLÜ ---")
negatif_satis = (df['Sales'] < 0).sum()
negatif_miktar = (df['Quantity'] < 0).sum()
print(f"Negatif değere sahip Satış sayısı: {negatif_satis}")
print(f"Negatif değere sahip Miktar sayısı: {negatif_miktar}")

# 3. Sayısal Verilerin İstatistiksel Özeti (Tansiyon Ölçümü)
print("\n--- İSTATİSTİKSEL ÖZET ---")
display(df.describe())

--- KOPYA SATIR KONTROLÜ ---
Tekrar eden satır sayısı: 0

--- MANTIKSAL HATA KONTROLÜ ---
Negatif değere sahip Satış sayısı: 0
Negatif değere sahip Miktar sayısı: 0

--- İSTATİSTİKSEL ÖZET ---


,Row_ID,Order_Date,Ship_Date,Sales,Quantity,Discount,Profit
count,9993.000000,9993,9993,9.993000e+03,9993.000000,9993.000000,9.993000e+03
mean,4997.265086,2013-04-30 17:54:59.549684,2013-05-04 16:55:28.730111,1.131662e+05,3.789353,1.750325,1.801138e+05
min,1.000000,2011-01-04 00:00:00,2011-01-08 00:00:00,3.000000e+00,1.000000,0.000000,-3.839990e+07
25%,2499.000000,2012-05-23 00:00:00,2012-05-27 00:00:00,2.516000e+03,2.000000,0.000000,2.245000e+03
50%,4997.000000,2013-06-27 00:00:00,2013-06-30 00:00:00,1.108800e+04,3.000000,2.000000,3.632200e+04
75%,7496.000000,2014-05-15 00:00:00,2014-05-19 00:00:00,5.399100e+04,5.000000,2.000000,1.509840e+05
max,9994.000000,2014-12-31 00:00:00,2015-01-06 00:00:00,2.396266e+07,14.000000,45.000000,6.719981e+07
std,2885.212415,NaN,NaN,5.186114e+05,2.225111,3.089019,1.723056e+06


In [7]:
# 1. AYLIK BÜYÜME ORANI VE HAREKETLİ ORTALAMA (Trend Analizi)
# Veriyi aylık bazda grupluyoruz
df_trend = df.copy()
df_trend['YearMonth'] = df_trend['Order_Date'].dt.to_period('M')
aylik_satislar = df_trend.groupby('YearMonth')['Sales'].sum().reset_index()

# Büyüme Oranını (Bir önceki aya göre % değişim) hesaplıyoruz
aylik_satislar['Growth_Rate_(%)'] = aylik_satislar['Sales'].pct_change() * 100

# 3 Aylık Hareketli Ortalama 
aylik_satislar['3_Month_Moving_Avg'] = aylik_satislar['Sales'].rolling(window=3).mean()

print("--- AYLIK TREND VE BÜYÜME ANALİZİ (İlk 5 Ay) ---")
display(aylik_satislar.head(6))


# 2. AYKIRI DEĞER (OUTLIER) ANALİZİ - İstatistiksel IQR Yöntemi
# Satışların %25'lik (Q1) ve %75'lik (Q3) kısımlarını buluyoruz
Q1 = df['Sales'].quantile(0.25)
Q3 = df['Sales'].quantile(0.75)
IQR = Q3 - Q1 # Orta kısımdaki yoğunluk

# Normal kabul edilen satışların alt ve üst sınırlarını belirliyoruz
alt_sinir = Q1 - 1.5 * IQR
ust_sinir = Q3 + 1.5 * IQR

# Sınırların dışında kalan siparişleri (Anormal Derecede Büyük/Küçük Satışlar) filtreliyoruz
outliers = df[(df['Sales'] < alt_sinir) | (df['Sales'] > ust_sinir)]

print("\n--- AYKIRI DEĞER (OUTLIER) KONTROLÜ ---")
print(f"Normal kabul edilen satış tutarı aralığı: {alt_sinir:.2f} ile {ust_sinir:.2f} arası.")
print(f"Bu aralığın dışına taşan (şüpheli/çok yüksek) sipariş sayısı: {len(outliers)}")

--- AYLIK TREND VE BÜYÜME ANALİZİ (İlk 5 Ay) ---


,YearMonth,Sales,Growth_Rate_(%),3_Month_Moving_Avg
0,2011-01,2617974.0,NaN,NaN
1,2011-02,1090093.0,-58.361198,NaN
2,2011-03,19889225.0,1724.543869,7.865764e+06
3,2011-04,8931683.0,-55.092856,9.970334e+06
4,2011-05,9349645.0,4.679544,1.272352e+07
5,2011-06,18297753.0,95.705324,1.219303e+07



--- AYKIRI DEĞER (OUTLIER) KONTROLÜ ---
Normal kabul edilen satış tutarı aralığı: -74696.50 ile 131203.50 arası.
Bu aralığın dışına taşan (şüpheli/çok yüksek) sipariş sayısı: 1486


In [8]:
# Temizlenmiş veriyi Power BI raporlaması için dışa aktarıyoruz
df.to_csv('cleaned_sales.csv', index=False)
print("Temizlenmiş veri 'cleaned_sales.csv' olarak başarıyla kaydedildi!")

Temizlenmiş veri 'cleaned_sales.csv' olarak başarıyla kaydedildi!
